In [ ]:
!pip install kaggle -q

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
#API to fetch the dataset from kaggle
!kaggle datasets download guy007/nutrientdeficiencysymptomsinrice

In [ ]:
# FIX: renamed 'dataset' variable to 'dataset_zip' to avoid conflict with the TF dataset defined later
from zipfile import ZipFile

dataset_zip = '/content/nutrientdeficiencysymptomsinrice.zip'
with ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall()
    print('Dataset extracted successfully.')

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from sklearn.metrics import confusion_matrix, classification_report
import pickle

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Configuration
BATCH_SIZE = 32
IMAGE_SIZE = 299   # IMPROVEMENT: Changed 256→299 (native InceptionV3 input size)
CHANNELS = 3
EPOCHS = 100
NUM_CLASSES = 3  # Nitrogen(N), Phosphorus(P), Potassium(K)
SEED = 42

DATA_DIR = "/content/rice_plant_lacks_nutrients"
selected_classes = ['Nitrogen(N)', 'Phosphorus(P)', 'Potassium(K)']

In [ ]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    seed=SEED,
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
    class_names=selected_classes
)

class_names = dataset.class_names
print(f"Classes: {class_names}")
print(f"Total batches: {len(dataset)}")

In [ ]:
# IMPROVEMENT: Check class distribution to spot imbalance
class_counts = {name: 0 for name in class_names}
for _, labels in dataset:
    for label in labels.numpy():
        class_counts[class_names[label]] += 1

print("Class distribution:")
for cls, count in class_counts.items():
    print(f"  {cls}: {count} images")

In [ ]:
plt.figure(figsize=(10, 10))
for image_batch, labels_batch in dataset.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(class_names[labels_batch[i]])
        plt.axis("off")
plt.suptitle("Sample Rice Plant Images", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def get_dataset_partitions_tf(ds, train_split=0.8, val_split=0.1, test_split=0.1,
                               shuffle=True, shuffle_size=10000):
    # FIX: Floating-point comparison replaced with round() to avoid AssertionError
    assert round(train_split + val_split + test_split, 5) == 1.0, \
        "Splits must sum to 1.0"

    ds_size = len(ds)

    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=SEED)

    train_size = int(train_split * ds_size)
    val_size   = int(val_split   * ds_size)

    train_ds = ds.take(train_size)
    val_ds   = ds.skip(train_size).take(val_size)
    # FIX: test_ds now correctly skips train+val (original skipped train twice)
    test_ds  = ds.skip(train_size + val_size)

    return train_ds, val_ds, test_ds


train_ds, val_ds, test_ds = get_dataset_partitions_tf(dataset)
print(f"Train batches : {len(train_ds)}")
print(f"Val batches   : {len(val_ds)}")
print(f"Test batches  : {len(test_ds)}")

In [ ]:
# IMPROVEMENT: Use InceptionV3's own preprocess_input instead of manual 1./255 rescaling
# This scales pixels to [-1, 1] as InceptionV3 was trained with

def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

# IMPROVEMENT: Augmentation applied ONLY to train set; val/test get only preprocessing
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.3),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.2),
], name="data_augmentation")

def augment_and_preprocess(image, label):
    image = data_augmentation(image, training=True)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

# Build optimised pipelines
train_ds = (train_ds
            .map(augment_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
            .cache()
            .shuffle(1000, seed=SEED)
            .prefetch(tf.data.AUTOTUNE))

val_ds = (val_ds
          .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
          .cache()
          .prefetch(tf.data.AUTOTUNE))

# FIX: test_ds should NOT be shuffled — order must be preserved for correct labels
test_ds = (test_ds
           .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
           .cache()
           .prefetch(tf.data.AUTOTUNE))

In [ ]:
# Load pretrained InceptionV3 backbone
inception_base = InceptionV3(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS),
    include_top=False,
    weights='imagenet'
)

# IMPROVEMENT: Two-phase freezing strategy
# Phase 1 — freeze entire backbone for initial training of head
inception_base.trainable = False
print(f"Total layers in InceptionV3: {len(inception_base.layers)}")

In [ ]:
# IMPROVEMENT: Use Functional API instead of Sequential for cleaner InceptionV3 integration
# IMPROVEMENT: GlobalAveragePooling2D instead of Flatten — reduces params and overfitting
inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
x = inception_base(inputs, training=False)   # training=False keeps BN layers frozen
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)            # IMPROVEMENT: BN after pooling stabilises training
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs, name="InceptionV3_RicePlant")
model.summary()

In [ ]:
# IMPROVEMENT: ReduceLROnPlateau + ModelCheckpoint added alongside EarlyStopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1
)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'best_model_inception.keras', monitor='val_accuracy', save_best_only=True, verbose=1
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),  # Higher LR for head-only phase
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print("=== Phase 1: Training classification head (backbone frozen) ===")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,           # Fewer epochs — head converges fast
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

In [ ]:
# IMPROVEMENT: Unfreeze last 50 layers for deeper fine-tuning
# (InceptionV3 has ~311 layers; unfreezing last 50 covers recent Inception modules)
inception_base.trainable = True
for layer in inception_base.layers[:-50]:
    layer.trainable = False

trainable_count = sum(1 for l in inception_base.layers if l.trainable)
print(f"Trainable InceptionV3 layers: {trainable_count} / {len(inception_base.layers)}")

# IMPROVEMENT: Much lower LR for fine-tuning to avoid destroying pretrained weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

early_stopping_ft = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)
reduce_lr_ft = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.3, patience=4, min_lr=1e-8, verbose=1
)

print("=== Phase 2: Fine-tuning top backbone layers ===")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping_ft, reduce_lr_ft, checkpoint],
    verbose=1
)

In [ ]:
# Load best saved weights
model = tf.keras.models.load_model('best_model_inception.keras')

scores = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss     : {scores[0]:.4f}")
print(f"Test Accuracy : {scores[1]*100:.2f}%")

In [ ]:
# Merge both phase histories for a single plot
def merge_history(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history_all = merge_history(history_phase1, history_phase2)
phase1_end  = len(history_phase1.history['accuracy'])

acc      = history_all['accuracy']
val_acc  = history_all['val_accuracy']
loss     = history_all['loss']
val_loss = history_all['val_loss']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, train_vals, val_vals, title in zip(
    axes,
    [acc, loss],
    [val_acc, val_loss],
    ['Accuracy', 'Loss']
):
    epochs_range = range(len(train_vals))
    ax.plot(epochs_range, train_vals, label=f'Train {title}')
    ax.plot(epochs_range, val_vals,   label=f'Val {title}')
    ax.axvline(x=phase1_end - 1, color='gray', linestyle='--', label='Fine-tune start')
    ax.set_title(f'Training & Validation {title}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
val_acc_final = history_all['val_accuracy'][-1]
val_error     = 100 - (val_acc_final * 100)
print(f"Validation Accuracy : {val_acc_final*100:.2f}%")
print(f"Validation Error    : {val_error:.2f}%")

In [ ]:
# FIX: Removed model.predict() inside loop (caused double-batching)
y_pred = []
y_true = []

for images, labels in test_ds:
    predictions = model(images, training=False)   # Direct call is faster
    predicted_labels = np.argmax(predictions.numpy(), axis=1)
    y_pred.extend(predicted_labels)
    y_true.extend(labels.numpy())

print(f"Total test samples: {len(y_true)}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=selected_classes,
            yticklabels=selected_classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# IMPROVEMENT: Per-class precision/recall/F1
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=selected_classes))

In [ ]:
from sklearn.metrics import f1_score

f1 = f1_score(y_true, y_pred, average='weighted')
print(f"Weighted F1 Score: {f1:.4f}")

In [ ]:
# FIX: predict() now applies InceptionV3's preprocess_input (was missing before)
def predict(model, img_array):
    """
    Predict class and confidence for a single image (H, W, C) numpy array.
    Expects raw uint8 pixel values [0, 255].
    """
    img = tf.cast(img_array, tf.float32)
    img = preprocess_input(img)
    img = tf.expand_dims(img, axis=0)        # Add batch dimension
    preds = model(img, training=False)
    idx   = np.argmax(preds[0])
    conf  = round(float(preds[0][idx]) * 100, 2)
    return class_names[idx], conf


# Reload one raw (un-preprocessed) batch for display
raw_display_ds = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    seed=SEED,
    shuffle=False,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=9,
    label_mode='int',
    class_names=selected_classes
)

plt.figure(figsize=(15, 15))
for images, labels in raw_display_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img_np = images[i].numpy().astype("uint8")
        plt.imshow(img_np)

        predicted_class, confidence = predict(model, images[i].numpy())
        actual_class = class_names[labels[i]]
        color = 'green' if predicted_class == actual_class else 'red'

        plt.title(
            f"Actual: {actual_class}\nPredicted: {predicted_class}\nConf: {confidence}%",
            color=color, fontsize=9
        )
        plt.axis("off")

plt.suptitle("Rice Plant Nutrient Deficiency Predictions (green=correct, red=wrong)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
model.save('inceptionV3_model.keras')
print("Model saved as 'inceptionV3_model.keras'")

# Save class names for inference
with open('class_names.pkl', 'wb') as f:
    pickle.dump(class_names, f)
print("Class names saved to class_names.pkl")